<a href="https://colab.research.google.com/github/rahmatnug/capstone-cangkringan-ml/blob/main/feature_engineering_timeseries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

print("Memulai Feature Engineering Time-Series...")

# 1. Load dataset sintetis terbaru (pastikan sudah berisi 6 SKU Riil)
df = pd.read_csv('synthetic_demand_data.csv')

# 2. Konversi tanggal dan pastikan urutan waktu benar (Crucial untuk TimeSeriesSplit)
df['tanggal_permintaan'] = pd.to_datetime(df['tanggal_permintaan'])
df = df.sort_values(by=['id_poktan', 'id_komoditas', 'tanggal_permintaan']).reset_index(drop=True)

# 3. Feature Engineering: Lag Features
# Karena data kita mingguan, lag 1 = 1 minggu, lag 4 = 1 bulan (sekitar 30 hari), lag 7 = 7 minggu.
df['lag_1w'] = df.groupby(['id_poktan', 'id_komoditas'])['volume_permintaan'].shift(1)
df['lag_4w'] = df.groupby(['id_poktan', 'id_komoditas'])['volume_permintaan'].shift(4) # Mendekati Lag 30 hari
df['lag_7w'] = df.groupby(['id_poktan', 'id_komoditas'])['volume_permintaan'].shift(7)

# 4. Feature Engineering: Rolling Statistics (REVISI DATA LEAKAGE)
# Mengambil rata-rata volume selama 4 minggu ke belakang TANPA menghitung target minggu ini
df['rolling_mean_4w'] = df.groupby(['id_poktan', 'id_komoditas'])['volume_permintaan'].transform(
    lambda x: x.shift(1).rolling(window=4).mean()
)

# 5. Handling Missing Values akibat Shifting
# Karena kita nge-shift data ke belakang (lag), baris-baris pertama pasti jadi NaN.
# Kita drop baris yang kosong ini agar XGBoost tidak error.
df_ready = df.dropna().reset_index(drop=True)

# 6. Ekspor dataset siap latih
df_ready.to_csv('features_demand_data.csv', index=False)

print(f"✅ Feature Engineering Selesai!")
print(f"Dataset awal: {len(df)} baris. Dataset siap latih (setelah drop NaN lag): {len(df_ready)} baris.")
print("Fitur baru yang ditambahkan: lag_1w, lag_4w, lag_7w, rolling_mean_4w")

Memulai Feature Engineering Time-Series...
✅ Feature Engineering Selesai!
Dataset awal: 2610 baris. Dataset siap latih (setelah drop NaN lag): 2400 baris.
Fitur baru yang ditambahkan: lag_1w, lag_4w, lag_7w, rolling_mean_4w
